<a href="https://colab.research.google.com/github/dudinha-web/fundamentos-de-ia/blob/main/Aula_06_Chat_PDF_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Aula 06 — Chat com PDF

Pipeline RAG usando LangChain

**Fluxo:**
1. Carregar PDF
2. Quebrar em chunks
3. Gerar embeddings
4. Guardar no banco vetorial (Chroma)
5. Retriever busca os trechos relevantes
6. LLM responde com base no contexto

In [ ]:
# Instalação das bibliotecas
!pip install -q \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-core \
    chromadb \
    sentence-transformers \
    transformers \
    pypdf \
    huggingface_hub \
    accelerate

In [ ]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline

print("Imports OK!")

In [ ]:
# 1. Carregar PDF
# Faça upload do seu PDF no Colab antes de executar esta célula
arquivo = '/content/pdf_1772.pdf'

loader = PyPDFLoader(arquivo)
documents = loader.load()



In [ ]:
print(f"Total de páginas carregadas: {len(documents)}")


In [ ]:
print("\nPrimeira página (trecho):")


In [ ]:
print(documents[0].page_content[:500])

In [ ]:
# 2. Quebrar texto em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=80
)
docs = text_splitter.split_documents(documents)

print(f"Total de chunks gerados: {len(docs)}")

In [ ]:
# 3. Embeddings + Banco vetorial + Retriever
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(docs, embeddings)

retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3}
)

print("Banco vetorial e retriever prontos!")

In [ ]:
# 4. LLM local

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200,
)

llm = HuggingFacePipeline(pipeline=pipe)

print("Modelo LLM carregado!")

In [ ]:
# 5. Prompt + Chain

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Pipeline LCEL: recupera → formata → prompt → llm → texto
chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Ok!")

In [ ]:
# 6. Loop de perguntas
print("Chat com PDF iniciado! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)

---
## Groq (gratuito)

1. Crie sua chave gratuita em: https://console.groq.com
2. E teste

In [ ]:
# Testando o GROK
!pip install -q langchain-groq

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Cole aqui sua chave Groq
os.environ["GROQ_API_KEY"] = ""

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_groq
    | StrOutputParser()
)

print("Chain com Groq pronta! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)